# Script 2 — Preparação & Engenharia de Features
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

In [ ]:

import pandas as pd
import numpy as np
import warnings, json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import Ridge

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)
print("✅ Dependências carregadas")


## 1. Carregar dataset consolidado

In [ ]:

dataset = pd.read_parquet(PASTA_SAIDA / 'dataset_cvm_consolidado.parquet')
print(f"Dataset carregado: {dataset.shape}")
print(f"Colunas: {list(dataset.columns[:15])}...")


## 2. Filtrar apenas DFP anuais (excluir ITR)

In [ ]:

# ETAPA 2: Manter apenas DFPs anuais — DT_REFER com mês 12 ou conforme fiscal
# Identifica exercícios anuais pelo gap entre DT_INI_EXERC e DT_FIM_EXERC ≈ 365 dias
# Simples: mantém somente 1 registro por empresa/ano (o mais completo)
dataset['DT_REFER'] = pd.to_datetime(dataset['DT_REFER'], errors='coerce')
dataset['ANO'] = dataset['DT_REFER'].dt.year

# Remove duplicatas mantendo o registro com maior cobertura de KPIs
KPIS = ['margem_bruta','margem_ebit','margem_liquida','margem_ebitda',
        'roe','roa','liquidez_corrente','liquidez_imediata',
        'endividamento','alavancagem_de','div_liquida','cobertura_juros',
        'giro_ativo','fco_receita','fco_lucro','EBITDA']
dataset['_n_kpis'] = dataset[KPIS].notna().sum(axis=1)
dataset = (dataset
    .sort_values(['CNPJ_CIA','ANO','_n_kpis'], ascending=[True,True,False])
    .drop_duplicates(subset=['CNPJ_CIA','ANO'])
    .drop(columns=['_n_kpis'])
    .sort_values(['CNPJ_CIA','ANO'])
    .reset_index(drop=True)
)
print(f"Após filtro anual: {dataset.shape}")
print(f"Empresas: {dataset['NOME_CIA'].nunique()} | Anos: {sorted(dataset['ANO'].unique())}")


## 3. Remoção de colunas com >80% de nulos

In [ ]:

# ETAPA 3: Remove colunas com >80% nulos
limiar_nulo = 0.80
cols_numericas = dataset.select_dtypes(include='number').columns
taxa_nulo = dataset[cols_numericas].isnull().mean()
cols_excluir = taxa_nulo[taxa_nulo > limiar_nulo].index.tolist()
print(f"Colunas excluídas (>{limiar_nulo:.0%} nulos): {len(cols_excluir)}")
dataset = dataset.drop(columns=cols_excluir)
print(f"Dataset após remoção: {dataset.shape}")


## 4. Imputação por mediana do setor

In [ ]:

# ETAPA 4: Imputa nulos pela mediana do setor, fallback mediana global
kpis_disponiveis = [k for k in KPIS if k in dataset.columns]
print(f"KPIs disponíveis para imputação: {len(kpis_disponiveis)}")

for kpi in kpis_disponiveis:
    # Mediana por setor
    mediana_setor = dataset.groupby('SETOR')[kpi].transform('median')
    # Mediana global
    mediana_global = dataset[kpi].median()
    # Imputação em cascata
    dataset[kpi] = (dataset[kpi]
        .fillna(mediana_setor)
        .fillna(mediana_global)
    )

taxa_nulo_pos = dataset[kpis_disponiveis].isnull().mean().mean()
print(f"Taxa de nulos após imputação: {taxa_nulo_pos:.2%}")


## 5. Winsorização de outliers (IQR × 3 por setor)

In [ ]:

# ETAPA 5: Winsorização por IQR × 3 por setor
def winsorizacao_setor(df, col, fator=3.0):
    def winsorizacao_grupo(grupo):
        Q1, Q3 = grupo[col].quantile(0.25), grupo[col].quantile(0.75)
        IQR = Q3 - Q1
        low, high = Q1 - fator * IQR, Q3 + fator * IQR
        grupo[col] = grupo[col].clip(lower=low, upper=high)
        return grupo
    return df.groupby('SETOR', group_keys=False).apply(winsorizacao_grupo)

for kpi in kpis_disponiveis:
    dataset = winsorizacao_setor(dataset, kpi)

print("✅ Winsorização aplicada")
print(f"Dataset: {dataset.shape}")


## 6. Variáveis de crescimento YoY (18 variáveis)

In [ ]:

# ETAPA 6: Variáveis de crescimento Year-over-Year
dataset = dataset.sort_values(['CNPJ_CIA','ANO']).reset_index(drop=True)

kpis_yoy = [k for k in kpis_disponiveis
            if k not in ['div_liquida']]  # excluir métricas absolutas de YoY

novas_cols_yoy = []
for kpi in kpis_yoy[:18]:  # até 18 variáveis YoY
    col_yoy = f'{kpi}_yoy'
    dataset[col_yoy] = (dataset.groupby('CNPJ_CIA')[kpi]
        .pct_change()
        .replace([np.inf, -np.inf], np.nan)
        .clip(-5, 5)  # limita a ±500%
    )
    novas_cols_yoy.append(col_yoy)

# Aceleração da receita
if 'margem_ebitda_yoy' in dataset.columns and 'margem_ebitda_yoy' in novas_cols_yoy:
    dataset['aceleracao_receita'] = dataset.groupby('CNPJ_CIA')['margem_ebitda_yoy'].diff()

print(f"Variáveis YoY criadas: {len(novas_cols_yoy)}")
print(f"Dataset: {dataset.shape}")


## 7. Codificação one-hot do setor

In [ ]:

# ETAPA 7: One-hot encoding do setor
dataset = pd.get_dummies(dataset, columns=['SETOR'], prefix='setor', dtype=float)
cols_setor = [c for c in dataset.columns if c.startswith('setor_')]
print(f"Colunas de setor criadas: {cols_setor}")


## 8. Criação dos targets por shift temporal

In [ ]:

# ETAPA 8: Targets = valor do KPI no próximo exercício (t+1)
# IMPORTANTE: shift dentro do grupo empresa, com anulação de lacunas temporais

TARGET_COLS = {
    'TARGET_DRE_3.01': 'DRE_3.01',  # Receita Líquida em t+1
    'TARGET_DRE_3.11': 'DRE_3.11',  # Lucro Líquido em t+1
    'TARGET_EBITDA':   'EBITDA',    # EBITDA em t+1
}

dataset = dataset.sort_values(['CNPJ_CIA','ANO']).reset_index(drop=True)

for target_col, source_col in TARGET_COLS.items():
    if source_col not in dataset.columns:
        print(f"⚠️  {source_col} não encontrado — target {target_col} não criado")
        continue
    # shift(-1) por empresa
    dataset[target_col] = dataset.groupby('CNPJ_CIA')[source_col].shift(-1)
    # Anula se há lacuna temporal > 1 ano
    ano_next = dataset.groupby('CNPJ_CIA')['ANO'].shift(-1)
    gap = ano_next - dataset['ANO']
    dataset.loc[gap != 1, target_col] = np.nan
    n_validos = dataset[target_col].notna().sum()
    print(f"  {target_col}: {n_validos} observações válidas")

# Remove última linha por empresa (não tem t+1)
dataset = dataset[dataset[list(TARGET_COLS.keys())].notna().any(axis=1)]
print(f"\nDataset com targets: {dataset.shape}")


## 9. Seleção de features por correlação e RFE

In [ ]:

# ETAPA 9: Seleção de features por correlação de Pearson + RFE com Ridge

FEATURES_BASE = kpis_disponiveis + novas_cols_yoy + cols_setor
if 'aceleracao_receita' in dataset.columns:
    FEATURES_BASE.append('aceleracao_receita')

FEATURES_BASE = [f for f in FEATURES_BASE if f in dataset.columns]

target_principal = 'TARGET_DRE_3.01'
if target_principal not in dataset.columns:
    print("⚠️  Target não encontrado — pule esta etapa")
else:
    df_sel = dataset[FEATURES_BASE + [target_principal]].dropna()
    X_sel  = df_sel[FEATURES_BASE]
    y_sel  = df_sel[target_principal]

    # Correlação de Pearson
    corr = X_sel.corrwith(y_sel).abs().sort_values(ascending=False)
    FEATURES_CORR = corr[corr > 0.1].index.tolist()  # threshold 0.1
    print(f"Features selecionadas por correlação: {len(FEATURES_CORR)}")

    # RFE com Ridge (top-15 features)
    if len(FEATURES_CORR) > 15:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_sel[FEATURES_CORR])
        rfe = RFE(Ridge(alpha=1.0), n_features_to_select=15, step=2)
        rfe.fit(X_scaled, y_sel)
        FEATURES_SELECIONADAS = [FEATURES_CORR[i] for i, s in enumerate(rfe.support_) if s]
    else:
        FEATURES_SELECIONADAS = FEATURES_CORR

    print(f"Features finais selecionadas: {len(FEATURES_SELECIONADAS)}")
    print(FEATURES_SELECIONADAS)


## 10. Split temporal treino/teste (75%/25%)

In [ ]:

# ETAPA 10: Split temporal por empresa, preservando ordem cronológica
# Cada empresa contribui com seus 75% mais antigos para treino

def split_temporal_empresa(grupo, frac_treino=0.75):
    n = len(grupo)
    n_treino = max(1, int(n * frac_treino))
    grupo = grupo.sort_values('ANO')
    grupo['split'] = 'teste'
    grupo.iloc[:n_treino, grupo.columns.get_loc('split')] = 'treino'
    return grupo

dataset = dataset.groupby('CNPJ_CIA', group_keys=False).apply(split_temporal_empresa)

treino = dataset[dataset['split'] == 'treino'].copy()
teste  = dataset[dataset['split'] == 'teste'].copy()

print(f"Treino: {len(treino)} obs | Teste: {len(teste)} obs")
print(f"  Razão treino: {len(treino)/(len(treino)+len(teste)):.1%}")
for setor in dataset['NOME_CIA'].unique()[:5]:
    t = dataset[dataset['NOME_CIA']==setor]
    print(f"  {setor[:20]:20s}: {(t['split']=='treino').sum()} treino / {(t['split']=='teste').sum()} teste")


## 11. Salvar artefatos para modelagem

In [ ]:

# ETAPA 11: Salva artefatos
import pickle

TARGETS = [t for t in TARGET_COLS if t in dataset.columns]

artefatos = {
    'dataset':             dataset,
    'treino':              treino,
    'teste':               teste,
    'features':            FEATURES_SELECIONADAS if 'FEATURES_SELECIONADAS' in dir() else FEATURES_BASE,
    'targets':             TARGETS,
    'kpis':                kpis_disponiveis,
    'cols_setor':          cols_setor,
}

for nome, obj in artefatos.items():
    if isinstance(obj, pd.DataFrame):
        obj.to_parquet(PASTA_SAIDA / f'{nome}.parquet', index=False)
    else:
        with open(PASTA_SAIDA / f'{nome}.pkl', 'wb') as f:
            pickle.dump(obj, f)

print("✅ Artefatos salvos:")
for nome, obj in artefatos.items():
    if isinstance(obj, pd.DataFrame):
        print(f"  {nome}.parquet: {obj.shape}")
    else:
        print(f"  {nome}.pkl: {obj}")

# Relatório de decisões
relatorio = {
    'n_observacoes_treino': len(treino),
    'n_observacoes_teste':  len(teste),
    'n_features':           len(artefatos['features']),
    'targets':              TARGETS,
    'kpis_disponiveis':     kpis_disponiveis,
    'limiar_nulo':          0.80,
    'fator_winsor':         3.0,
    'frac_treino':          0.75,
}
with open(PASTA_SAIDA / 'relatorio_preparacao.json', 'w', encoding='utf-8') as f:
    import json
    json.dump(relatorio, f, indent=2, ensure_ascii=False)
print("\n✅ Relatório de decisões salvo.")
